In [1]:
import h5py
import numpy

In [ ]:
f = h5py.File("file5.hdf5", 'w') # CREATE FILE
dset = f.create_dataset("my dataset", (10, 10), dtype='f') # CREATE DATASET/ WRITE VALUES

In [3]:
dset.shape

(10, 10)

In [ ]:
dset[0,0] = 2.0 # CHANGE VALUES

In [5]:
dset

<HDF5 dataset "my dataset": shape (10, 10), type "<f4">

In [6]:
dset[0,0]

np.float32(2.0)

In [7]:
dset[0,1]

np.float32(0.0)

In [8]:
f.close()

## Partial I/O

In [9]:
example_data = numpy.arange(100).reshape((10,10))
example_data

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
       [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
       [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
       [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
       [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
       [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
       [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]])

In [10]:
f = h5py.File('partial_io.hdf5', 'w')
dset = f.create_dataset('numbers', data=example_data)

In [11]:
dset.shape

(10, 10)

In [12]:
type(dset) # ofcourse, this is not an NpArray

h5py._hl.dataset.Dataset

In [13]:
f = h5py.File('partial_io.hdf5', 'r')

In [16]:
f['numbers'][:]

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
       [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
       [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
       [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
       [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
       [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
       [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]])

In [17]:
f.close()

In [25]:
dset[4,5]

np.int64(45)

## Chunks

In [49]:
f = h5py.File("file5.hdf5", 'w') # CREATE FILE
dset = f.create_dataset("chunked_dataset", (10, 10), dtype='f', chunks=(5, 5), data=numpy.arange(100).reshape((10,10))) # CREATE DATASET

In [50]:
f.close()

In [51]:
f = h5py.File("file5.hdf5", 'r') # CREATE FILE
dset=f["chunked_dataset"]

In [54]:
for s in dset.iter_chunks():
    print(dset[s])

[[ 0.  1.  2.  3.  4.]
 [10. 11. 12. 13. 14.]
 [20. 21. 22. 23. 24.]
 [30. 31. 32. 33. 34.]
 [40. 41. 42. 43. 44.]]
[[ 5.  6.  7.  8.  9.]
 [15. 16. 17. 18. 19.]
 [25. 26. 27. 28. 29.]
 [35. 36. 37. 38. 39.]
 [45. 46. 47. 48. 49.]]
[[50. 51. 52. 53. 54.]
 [60. 61. 62. 63. 64.]
 [70. 71. 72. 73. 74.]
 [80. 81. 82. 83. 84.]
 [90. 91. 92. 93. 94.]]
[[55. 56. 57. 58. 59.]
 [65. 66. 67. 68. 69.]
 [75. 76. 77. 78. 79.]
 [85. 86. 87. 88. 89.]
 [95. 96. 97. 98. 99.]]


## Corrupted data

In [16]:
import h5py
import numpy as np
with h5py.File('corrupt.h5', 'r') as f:
    chunks_size = f['cells'].chunks
    dset = f['cells'][:]

In [17]:
np.shape(dset)
print(chunks_size)

(16, 16, 16)


In [19]:
test_image = dset[:]
shape = test_image.shape
cz, cy, cx = chunks_size

bad_chunks = []

for z in range(0, shape[0], cz):
    for y in range(0, shape[1], cy):
        for x in range(0, shape[2], cx):
            try:
                _ = test_image[z:z+cz, y:y+cy, x:x+cx]
            except Exception:
                bad_chunks.append((z, y, x))

print("Corrupted chunks:", bad_chunks)

Corrupted chunks: []


In [20]:
import napari

viewer = napari.Viewer()
viewer.add_image(test_image)
napari.run()

## Finding the filters

In [2]:
import h5py
from watermasks.utils import paths
import hdf5plugin

path_h5, path_xml, path_mast, dir_input_images, dir_output_masks = paths('JR_22-11-17_deconvolved')

In [ ]:
with h5py.File(path_h5, "r") as f:
    dset = f[f't00400'][f's02']['2']['cells']

    print("compression:", dset.compression)
    print("compression_opts:", dset.compression_opts)
    print(i for i in dset.attrs)
    print(j for i in dset.asstr for j in i)
    for k,j in dset.attrs.items():
        print(k,'=', j)

compression: gzip
compression_opts: 6
<generator object <genexpr> at 0x000001D2E182B1B0>


In [7]:
with h5py.File(path_h5, 'r') as f:
    dset = f[f't00300'][f's02']['2']['cells']
    print(dset._filters)

{'scaleoffset': (2, 0, 4096, 0, 2, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1818321779, 1717989221, 7628147, 0), 'gzip': 6}


In [3]:
with h5py.File(path_h5, 'r') as f:
    dset = f[f't00400'][f's02']['2']['cells']
    print(dset._filters)
    print(dset.compression)

{'scaleoffset': (2, 0, 4096, 0, 2, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1818321779, 1717989221, 7628147, 0), 'gzip': 6}
gzip
